# LogSense — Data Preprocessing
This notebook prepares the cybersecurity dataset for machine learning.
We will clean, encode, split and scale the data so it is ready for model training.

In [21]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/cybersecurity_intrusion_data.csv')
print(f"Data loaded — {df.shape[0]} rows, {df.shape[1]} columns")

Data loaded — 9537 rows, 11 columns


## Step 1 — Handle Missing Values
`encryption_used` has 1,966 missing values. These represent sessions 
with no encryption at all, so we fill them with the string "None" 
rather than dropping the rows entirely.

In [22]:
# Fill missing encryption_used values with "None"
df['encryption_used'] = df['encryption_used'].fillna('None')

# Confirm no more missing values
print("Missing values after fix:")
print(df.isnull().sum())

Missing values after fix:
session_id             0
network_packet_size    0
protocol_type          0
login_attempts         0
session_duration       0
encryption_used        0
ip_reputation_score    0
failed_logins          0
browser_type           0
unusual_time_access    0
attack_detected        0
dtype: int64


## Step 2 — Encode Text Columns
Machine learning models only understand numbers, not text.
We convert text columns (protocol_type, encryption_used, browser_type)
into numbers using one-hot encoding.

In [23]:
# Drop session_id — it's just an identifier, not useful for the model
df = df.drop(columns=['session_id'])

# One-hot encode the text columns
df_encoded = pd.get_dummies(df, columns=['protocol_type', 'encryption_used', 'browser_type'])

print(f"Shape before encoding: {df.shape}")
print(f"Shape after encoding: {df_encoded.shape}")
print("\nNew columns:")
print(df_encoded.columns.tolist())

Shape before encoding: (9537, 10)
Shape after encoding: (9537, 18)

New columns:
['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score', 'failed_logins', 'unusual_time_access', 'attack_detected', 'protocol_type_ICMP', 'protocol_type_TCP', 'protocol_type_UDP', 'encryption_used_AES', 'encryption_used_DES', 'encryption_used_None', 'browser_type_Chrome', 'browser_type_Edge', 'browser_type_Firefox', 'browser_type_Safari', 'browser_type_Unknown']


## Step 3 — Split the Data
We split into features (X) and target (y), then divide into
80% training data and 20% test data.
The model learns from the training set and gets evaluated on the test set.

In [24]:
from sklearn.model_selection import train_test_split

# X = everything except the target, y = the target
X = df_encoded.drop(columns=['attack_detected'])
y = df_encoded['attack_detected']

# 80% train, 20% test, random_state makes it reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining target distribution:")
print(y_train.value_counts())

Training set: (7629, 17)
Test set: (1908, 17)

Training target distribution:
attack_detected
0    4231
1    3398
Name: count, dtype: int64


## Step 4 — Scale the Numerical Features
We scale number columns so they are all on the same scale.
This prevents large numbers from dominating the model unfairly.

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit on training data only, then transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete!")
print(f"Training set mean (should be ~0): {X_train_scaled.mean().round(2)}")
print(f"Training set std (should be ~1): {X_train_scaled.std().round(2)}")

Scaling complete!
Training set mean (should be ~0): -0.0
Training set std (should be ~1): 1.0


In [27]:
import os

# Create model folder if it doesn't exist
os.makedirs('../model', exist_ok=True)
print("model/ folder created!")

model/ folder created!


In [28]:
import joblib

# Save the scaler so we can reuse it later for new data
joblib.dump(scaler, '../model/scaler.pkl')

# Save the processed data
np.save('../model/X_train_scaled.npy', X_train_scaled)
np.save('../model/X_test_scaled.npy', X_test_scaled)
np.save('../model/y_train.npy', y_train)
np.save('../model/y_test.npy', y_test)

print("All files saved to model/ folder!")

All files saved to model/ folder!
